# SHAP Explainability for an SVM Classifier

This notebook demonstrates how to use **SHAP (SHapley Additive exPlanations)** to interpret the predictions of a nonlinear **Support Vector Machine (SVM)** with an **RBF kernel**.

The dataset used in this notebook is the OpenML dataset:

- **Dataset name:** `heart-disease`
- **OpenML data_id:** `43672`

The target variable is binary:

- `0` = no heart disease
- `1` = heart disease

SHAP explanations are computed on the probability estimates produced by the trained SVM classifier.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV

import shap

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Import Dataset

The code below:

- reads the dataset from a CSV file
- defines the categorical and numerical features
- creates:
  - `X`: dataframe containing the input features
  - `y`: target array (`1 = disease`, `0 = no disease`)
  - `numerical_features`: list of numerical variables
  - `categorical_features`: list of categorical variables

In [ ]:
d = pd.read_csv('data.csv')
categorical_features = ['sex', 'chest_pain_type', 'fasting_blood_sugar', 'resting_ecg', 'exercise_angina', 'ST_slope']
numerical_features = ['age', 'resting_bp_s', 'cholesterol', 'max_heart_rate', 'oldpeak']
X = d[categorical_features + numerical_features]
y = d['target'].values

## 3. Definition of the Machine Learning Pipeline

* Create a pipeline  for numerical features that:
  * imputes missing values using the median (Use SimpleImputer)
  * standardizes features using `StandardScaler`
      * mean = 0
      * standard deviation = 1

* Create a second pipeline  for categorical features that:
  * imputes missing values using the most frequent category (Use SimpleImputer)
  * applies one-hot encoding (Use OneHotEncoder)

* Combine the two preprocessing pipelines using `ColumnTransformer`, allowing different transformations to be applied to different feature groups

* Split the dataset into training and testing using stratified sampling to preserve the class distribution.

* Perform a grid search with 5-fold cross-validation to compare linear SVM and RBF-kernel SVM, using the following values for C = np.logspace(-4, 4, 9) and for gamma = np.logspace(-4, 1, 6)

## 5. Create the SHAP Explainer

SHAP explanations require a **background dataset** representing the typical distribution of the training data.

Here, the background dataset is sampled from the **raw training dataframe**.

The explainer is then created using:

```python
best_model.predict_proba
```

This means SHAP calls the full fitted pipeline each time it needs a prediction:

1. raw input data are passed to the pipeline
2. the preprocessing step transforms the data
3. the SVM classifier returns class probabilities


The background dataset is used to estimate the baseline prediction $E[f(X)]$ which represents the average model output over the background samples.

In [ ]:
# Sample background data from the raw training dataframe
background = shap.sample(
    X_train,
    50,
    random_state=RANDOM_STATE
)

# Explain the full fitted pipeline directly
explainer = shap.Explainer(
    best_model.predict_proba,
    background
)

## 6. Compute SHAP Values

This section computes SHAP values for a subset of the raw test samples.

The code:

- selects the first test samples to explain
- passes the raw samples to the full fitted pipeline through the SHAP explainer
- computes feature contribution scores for each original input feature and each class

Because the explainer wraps the complete pipeline, the SHAP values are reported with respect to the original input columns rather than the one-hot encoded internal representation.

The resulting SHAP tensor usually has shape: $(number\_of\_samples,\ number\_of\_original\_features,\ number\_of\_classes)$

For example:

```python
shap_values[0, :, 1]
```

returns the SHAP values for:

- sample `0`
- all original input features
- class `1` (heart disease)

Positive SHAP values increase the probability of the selected class, while negative SHAP values decrease it.

In [ ]:
n_explain = 10

# Select raw test samples to explain
X_explain = X_test.iloc[:n_explain]

# Compute SHAP values using the full fitted pipeline
shap_values = explainer(X_explain)

print("SHAP values shape:", shap_values.values.shape)
print("Base values shape:", shap_values.base_values.shape)
print("Explained data shape:", X_explain.shape)

## 7. Global Feature Importance with a Beeswarm Plot

The beeswarm plot provides a global view of feature importance across the explained samples.

This section:

- extracts SHAP values corresponding to the positive class (`class 1`)
- creates a `shap.Explanation` object for the positive class
- uses the original feature values and feature names
- visualizes the distribution of SHAP values using a beeswarm plot

Interpretation:

- features are ordered by average importance
- each point corresponds to one sample
- horizontal position represents the SHAP contribution
- red points generally indicate high feature values
- blue points generally indicate low feature values

Because the full pipeline is explained directly, the displayed features correspond to the original dataset columns, not the internal one-hot encoded columns.

In [ ]:
positive_class_index = 1

shap_values_positive = shap.Explanation(
    values=shap_values.values[:, :, positive_class_index],
    base_values=shap_values.base_values[:, positive_class_index],
    data=X_explain.values,
    feature_names=X_explain.columns.tolist()
)

shap.plots.beeswarm(shap_values_positive)

## 8. Local Explanation with a Waterfall Plot

The waterfall plot explains the prediction for a single sample.

This section:

- selects one raw test sample from the explained subset
- extracts the SHAP values for the positive class
- builds a single-sample `shap.Explanation` object
- visualizes how each original feature contributes to the final predicted probability

The waterfall plot starts from the baseline prediction $E[f(X)]$ and then shows how individual feature contributions push the prediction until reaching the final probability predicted by the full pipeline.

In [ ]:
sample_idx = 1
positive_class_index = 1

single_sample_explanation = shap.Explanation(
    values=shap_values.values[sample_idx, :, positive_class_index],
    base_values=shap_values.base_values[sample_idx, positive_class_index],
    data=X_explain.iloc[sample_idx].values,
    feature_names=X_explain.columns.tolist()
)

shap.plots.waterfall(single_sample_explanation)